# CelebA-HQ — read-only path index for the mounted DIRE tree

Attach the existing full DIRE Kaggle Dataset as input, then run the definition cell and the index cell. Keep Accelerator Off. The index scans only `train/celebahq` and the selected `test/celebahq` groups.

It does not modify `/kaggle/input`, copy data, open/decode images, compute hashes, create validation splits, or train. It writes one JSON index under `/kaggle/working`. The member-style locator is inferred from the extracted tree and is explicitly not verified against original tar archives.

Stop after reviewing the counts and locator fields. Do not run a training notebook from this path index alone.

## Stage 2 — SHA-256 + full image decode (opt-in)

Chỉ chạy stage này sau khi path-index hoàn tất. Nó đọc từng file hai lượt (SHA-256, sau đó giải mã đầy đủ bằng Pillow), nhưng không ghi vào input và không dùng GPU. Checkpoint SQLite được lưu ngoài input tại `/kaggle/working`; nếu cell bị ngắt trong cùng runtime, chạy lại để resume. Trước khi kết thúc phiên, hãy lưu/export checkpoint và report nếu cần tiếp tục ở phiên khác.

Locator vẫn là suy luận từ cây đã giải nén; SHA-256 xác định bytes hiện tại, không xác nhận archive gốc/provenance.


In [ ]:
"""Read-only inventory adapter for the extracted CelebA-HQ DIRE maps on Kaggle.

The Kaggle input reported by the user is an extracted directory tree, e.g.
``dire/train/celebahq/real/real/<image>``. This module resolves the repeated
leaf folder without copying or changing the mounted data. The inferred member
locator is explicitly not proof of the original tar member path; archive bytes
and provenance are unavailable from this extracted tree.

This is an inventory/path adapter only. It does not assign validation splits,
open/decode images, compute hashes, train, or modify input data.
"""

from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path

IMAGE_SUFFIXES = {".bmp", ".jpeg", ".jpg", ".png", ".tif", ".tiff", ".webp"}
REQUIRED_GROUPS: tuple[tuple[str, str], ...] = (
    ("train", "real"),
    ("train", "sdv2"),
    ("test", "real"),
    ("test", "dalle2"),
    ("test", "if"),
    ("test", "midjourney"),
    ("test", "stargan"),
    ("test", "sdv2"),
)


class CelebAHQTreeLayoutError(ValueError):
    """The extracted tree is missing a required group or has ambiguous layout."""


@dataclass(frozen=True)
class CelebAHQImage:
    """One image path and its class metadata; no image bytes are read."""

    split: str
    generator: str
    label: int
    path: Path
    relative_path: str
    member_locator: str
    locator_inferred: bool
    leaf_layout: str


def _iter_image_files(directory: Path):
    """Yield regular image files in stable order without following symlinks."""
    for current, dirnames, filenames in os.walk(directory, topdown=True, followlinks=False):
        current_path = Path(current)
        dirnames[:] = sorted(
            name for name in dirnames
            if not (current_path / name).is_symlink()
        )
        for filename in sorted(filenames):
            path = current_path / filename
            if path.is_symlink() or path.suffix.lower() not in IMAGE_SUFFIXES:
                continue
            if path.is_file():
                yield path


def _direct_images(directory: Path) -> list[Path]:
    return sorted(
        path for path in directory.iterdir()
        if path.is_file()
        and not path.is_symlink()
        and path.suffix.lower() in IMAGE_SUFFIXES
    )


def scan_celebahq_tree(root: str | Path) -> list[CelebAHQImage]:
    """Index the selected CelebA-HQ train/test classes from an extracted tree.

    ``root`` is the mounted ``dire`` directory. For each generator folder, the
    scanner prefers its repeated same-name child (``real/real``) when it holds
    images and the outer folder has no direct images. If images exist both
    directly and below the repeated child, it raises instead of silently
    choosing one. A flat ``real/<image>`` layout is also accepted.

    ``member_locator`` approximates a full member-style locator such as
    ``real/0001.png`` by prefixing the generator name to the path under the
    selected leaf. ``locator_inferred`` remains true because this dataset is
    extracted and the original tar archives were not verified in this scan.
    """
    root_path = Path(root).expanduser().resolve()
    if not root_path.is_dir():
        raise NotADirectoryError(f"DIRE root is not a directory: {root_path}")

    records: list[CelebAHQImage] = []
    missing: list[str] = []
    for split, generator in REQUIRED_GROUPS:
        outer = root_path / split / "celebahq" / generator
        if not outer.is_dir():
            missing.append(f"{split}/celebahq/{generator}")
            continue

        direct = _direct_images(outer)
        repeated = outer / generator
        repeated_images = (
            list(_iter_image_files(repeated))
            if repeated.is_dir() and not repeated.is_symlink()
            else []
        )
        other_nested_images = []
        for child in sorted(outer.iterdir()):
            if child == repeated or not child.is_dir() or child.is_symlink():
                continue
            other_nested_images.extend(_iter_image_files(child))
        if other_nested_images:
            raise CelebAHQTreeLayoutError(
                f"unexpected nested image directories under {outer}: "
                f"{[path.relative_to(outer).as_posix() for path in other_nested_images[:5]]}"
            )
        if direct and repeated_images:
            raise CelebAHQTreeLayoutError(
                f"ambiguous class directory {outer}: found direct images and images under {repeated}"
            )
        if repeated_images:
            leaf_root = repeated
            leaf_layout = "repeated-generator-child"
            files = repeated_images
        elif direct:
            leaf_root = outer
            leaf_layout = "flat-class-directory"
            files = direct
        else:
            missing.append(f"{split}/celebahq/{generator} (no image files in expected leaf)")
            continue

        for path in files:
            relative = path.relative_to(root_path).as_posix()
            member_inside_leaf = path.relative_to(leaf_root).as_posix()
            member_locator = (Path(generator) / member_inside_leaf).as_posix()
            records.append(CelebAHQImage(
                split=split,
                generator=generator,
                label=0 if generator == "real" else 1,
                path=path,
                relative_path=relative,
                member_locator=member_locator,
                locator_inferred=True,
                leaf_layout=leaf_layout,
            ))

    if missing:
        raise CelebAHQTreeLayoutError(
            "missing or empty required CelebA-HQ groups: " + ", ".join(missing)
        )
    return records


In [ ]:
# Read-only path index: only the selected CelebA-HQ branches, no GPU/data copy
import json
from collections import Counter
from pathlib import Path

# Exact mounted DIRE root from the submitted Kaggle audit; edit only if Kaggle's mount path changes.
DIRE_ROOT = Path('/kaggle/input/datasets/j4cknguyn/diffusionforensics/dire')
REPORT_PATH = Path('/kaggle/working/celebahq-path-index.json')

records = scan_celebahq_tree(DIRE_ROOT)
counts = Counter((row.split, row.generator) for row in records)
report = {
    'schema_version': 1,
    'status': 'complete',
    'root': str(DIRE_ROOT),
    'purpose': 'CelebA-HQ path index only; no pixel read/decode, hash, copy, or training',
    'input_modified': False,
    'image_contents_read': False,
    'gpu_work_started': False,
    'archive_member_paths_verified': False,
    'locator_policy': 'inferred by prefixing generator to path relative to selected leaf directory',
    'validation_split_assigned': False,
    'counts': [
        {'split': split, 'generator': generator, 'label': 0 if generator == 'real' else 1, 'files': n}
        for (split, generator), n in sorted(counts.items())
    ],
    'records': [
        {
            'split': row.split,
            'generator': row.generator,
            'label': row.label,
            'path_from_dire_root': row.relative_path,
            'member_locator_inferred': row.member_locator,
            'locator_inferred': row.locator_inferred,
            'leaf_layout': row.leaf_layout,
        }
        for row in records
    ],
    'notes': [
        'Counts are regular image files recognized by suffix; images were not opened or decoded.',
        'member_locator_inferred is a stable path inference, not a verified original tar member path or source identity.',
        'Only train/test CelebA-HQ groups are indexed; LSUN and ImageNet are left untouched.',
    ],
}
REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print('Index status:', report['status'])
print('Indexed files:', len(records))
for row in report['counts']:
    print(f"  {row['split']:5s} {row['generator']:12s} label={row['label']} n={row['files']}")
print('Manifest:', REPORT_PATH)


In [ ]:
"""Read-only SHA-256 and image-decode audit for the selected CelebA-HQ tree.

The mounted input is never changed. Checkpoint database and exported manifests
are written only beneath a caller-provided writable directory outside the input.
The SQLite checkpoint can resume completed rows for the same file-tree metadata
fingerprint (root-relative paths, sizes, and mtimes).
"""

from __future__ import annotations

import hashlib
import json
import os
import sqlite3
import time
from pathlib import Path
from typing import Any, Iterable

CHUNK_SIZE = 1024 * 1024


def _inside(path: Path, root: Path) -> bool:
    return path == root or root in path.parents


def _source_fingerprint(records: list[Any], root: Path) -> str:
    digest = hashlib.sha256()
    for record in sorted(records, key=lambda row: row.relative_path):
        path = Path(record.path).expanduser().resolve()
        if not _inside(path, root):
            raise ValueError(f"indexed input path escapes DIRE root: {record.relative_path}")
        stat = path.stat()
        for value in (
            record.relative_path,
            record.split,
            record.generator,
            str(int(record.label)),
            record.member_locator,
            str(stat.st_size),
            str(stat.st_mtime_ns),
        ):
            digest.update(value.encode("utf-8"))
            digest.update(b"\0")
        digest.update(b"\n")
    digest.update(str(root).encode("utf-8"))
    return digest.hexdigest()


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(CHUNK_SIZE), b""):
            digest.update(block)
    return digest.hexdigest()


def _init_db(db: sqlite3.Connection, *, root: Path, fingerprint: str, count: int) -> None:
    db.execute("CREATE TABLE IF NOT EXISTS meta (key TEXT PRIMARY KEY, value TEXT NOT NULL)")
    db.execute(
        """CREATE TABLE IF NOT EXISTS files (
            relative_path TEXT PRIMARY KEY,
            split TEXT NOT NULL,
            generator TEXT NOT NULL,
            label INTEGER NOT NULL,
            member_locator TEXT NOT NULL,
            locator_inferred INTEGER NOT NULL,
            size_bytes INTEGER NOT NULL,
            mtime_ns INTEGER NOT NULL,
            sha256 TEXT,
            hash_status TEXT NOT NULL,
            decode_status TEXT NOT NULL,
            image_format TEXT,
            width INTEGER,
            height INTEGER,
            mode TEXT,
            error TEXT
        )"""
    )
    expected = {
        "schema_version": "1",
        "root": str(root),
        "source_fingerprint": fingerprint,
        "record_count": str(count),
    }
    existing = dict(db.execute("SELECT key, value FROM meta"))
    if existing and existing != expected:
        raise ValueError("checkpoint metadata mismatch; refusing to reuse another input manifest")
    if not existing:
        db.executemany("INSERT INTO meta(key, value) VALUES (?, ?)", expected.items())


def verify_celebahq_records(
    root: str | Path,
    records: Iterable[Any],
    workdir: str | Path,
    *,
    checkpoint_every: int = 250,
    progress_every: int = 1000,
) -> dict[str, Any]:
    """Hash and fully decode each indexed file, with resumable external checkpoint.

    Records are expected to expose ``split``, ``generator``, ``label``, ``path``,
    ``relative_path``, ``member_locator``, and ``locator_inferred`` (as emitted by
    :func:`scan_celebahq_tree`). Hashing and Pillow decoding are separate reads.
    This verifies the currently mounted bytes, not original archive provenance.
    """
    if checkpoint_every < 1 or progress_every < 1:
        raise ValueError("checkpoint_every and progress_every must be positive")

    try:
        from PIL import Image
    except ImportError as exc:  # pragma: no cover - Kaggle normally provides Pillow
        raise RuntimeError("Pillow is required for image decode verification") from exc

    root_path = Path(root).expanduser().resolve()
    if not root_path.is_dir():
        raise NotADirectoryError(f"DIRE root is not a directory: {root_path}")
    rows = list(records)
    if not rows:
        raise ValueError("no indexed CelebA-HQ records were supplied")
    work_path = Path(workdir).expanduser().resolve()
    if _inside(work_path, root_path):
        raise ValueError("checkpoint/output directory must be outside the read-only input dataset")

    fingerprint = _source_fingerprint(rows, root_path)
    run_id = fingerprint[:20]
    work_path.mkdir(parents=True, exist_ok=True)
    db_path = work_path / f"celebahq-integrity-{run_id}.sqlite3"
    manifest_path = work_path / f"celebahq-integrity-{run_id}.jsonl"
    summary_path = work_path / f"celebahq-integrity-{run_id}-summary.json"

    db = sqlite3.connect(db_path)
    try:
        _init_db(db, root=root_path, fingerprint=fingerprint, count=len(rows))
        db.commit()
        hashed_now = 0
        reused = 0
        pending = 0
        started = time.time()

        for index, record in enumerate(rows, start=1):
            path = Path(record.path)
            stat = path.stat()
            previous = db.execute(
                "SELECT size_bytes, mtime_ns, sha256, hash_status, decode_status "
                "FROM files WHERE relative_path = ?",
                (record.relative_path,),
            ).fetchone()
            if (
                previous
                and previous[0] == stat.st_size
                and previous[1] == stat.st_mtime_ns
                and previous[2]
                and previous[3] == "pass"
                and previous[4] in {"pass", "fail"}
            ):
                reused += 1
            else:
                hash_status = "pass"
                sha256 = None
                decode_status = "fail"
                image_format = None
                width = height = None
                mode = None
                errors: list[str] = []
                try:
                    sha256 = _sha256_file(path)
                except OSError as exc:
                    hash_status = "fail"
                    errors.append(f"sha256: {exc}")
                try:
                    with Image.open(path) as image:
                        image_format = image.format
                        width, height = image.size
                        mode = image.mode
                        image.load()  # force full pixel decode; do not retain pixel data
                    decode_status = "pass"
                except Exception as exc:  # record bad files and continue through the inventory
                    errors.append(f"decode: {type(exc).__name__}: {exc}")

                db.execute(
                    """INSERT OR REPLACE INTO files(
                        relative_path, split, generator, label, member_locator,
                        locator_inferred, size_bytes, mtime_ns, sha256, hash_status,
                        decode_status, image_format, width, height, mode, error
                    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""",
                    (
                        record.relative_path,
                        record.split,
                        record.generator,
                        int(record.label),
                        record.member_locator,
                        int(record.locator_inferred),
                        stat.st_size,
                        stat.st_mtime_ns,
                        sha256,
                        hash_status,
                        decode_status,
                        image_format,
                        width,
                        height,
                        mode,
                        " | ".join(errors)[:1000] or None,
                    ),
                )
                hashed_now += 1
                pending += 1
                if pending >= checkpoint_every:
                    db.commit()
                    pending = 0

            if index % progress_every == 0 or index == len(rows):
                db.commit()
                print(
                    f"integrity {index}/{len(rows)}; hashed this run={hashed_now}; "
                    f"reused checkpoint={reused}; elapsed={time.time() - started:.1f}s"
                )

        db.commit()
        total = db.execute("SELECT COUNT(*) FROM files").fetchone()[0]
        total_file_bytes = db.execute("SELECT COALESCE(SUM(size_bytes), 0) FROM files").fetchone()[0]
        hash_failures = db.execute("SELECT COUNT(*) FROM files WHERE hash_status != 'pass'").fetchone()[0]
        decode_failures = db.execute("SELECT COUNT(*) FROM files WHERE decode_status != 'pass'").fetchone()[0]
        missing_hashes = db.execute("SELECT COUNT(*) FROM files WHERE sha256 IS NULL").fetchone()[0]
        group_counts = [
            {
                "split": split,
                "generator": generator,
                "label": label,
                "files": count,
                "hash_failures": hash_errors,
                "decode_failures": decode_errors,
            }
            for split, generator, label, count, hash_errors, decode_errors in db.execute(
                """SELECT split, generator, label, COUNT(*),
                          SUM(CASE WHEN hash_status != 'pass' THEN 1 ELSE 0 END),
                          SUM(CASE WHEN decode_status != 'pass' THEN 1 ELSE 0 END)
                   FROM files GROUP BY split, generator, label
                   ORDER BY split, generator"""
            )
        ]

        temporary_manifest = manifest_path.with_suffix(manifest_path.suffix + ".tmp")
        metadata = {
            "record_type": "metadata",
            "schema_version": 1,
            "root": str(root_path),
            "source_fingerprint": fingerprint,
            "source_fingerprint_basis": "relative path + stat size + mtime_ns; no content identity",
            "archive_member_paths_verified": False,
            "input_modified": False,
            "gpu_work_started": False,
            "records": total,
        }
        with temporary_manifest.open("w", encoding="utf-8") as stream:
            stream.write(json.dumps(metadata, ensure_ascii=False) + "\n")
            cursor = db.execute(
                """SELECT relative_path, split, generator, label, member_locator,
                          locator_inferred, size_bytes, sha256, hash_status,
                          decode_status, image_format, width, height, mode, error
                   FROM files ORDER BY split, generator, relative_path"""
            )
            for row in cursor:
                payload = dict(zip((
                    "path_from_dire_root", "split", "generator", "label",
                    "member_locator_inferred", "locator_inferred", "size_bytes",
                    "sha256", "hash_status", "decode_status", "image_format",
                    "width", "height", "mode", "error",
                ), row))
                payload["locator_inferred"] = bool(payload["locator_inferred"])
                stream.write(json.dumps(payload, ensure_ascii=False) + "\n")
        os.replace(temporary_manifest, manifest_path)

        summary: dict[str, Any] = {
            "schema_version": 1,
            "status": "pass" if total == len(rows) and not hash_failures and not decode_failures else "fail",
            "root": str(root_path),
            "source_fingerprint": fingerprint,
            "source_fingerprint_basis": "relative path + stat size + mtime_ns; no content identity",
            "input_modified": False,
            "image_contents_read": True,
            "gpu_work_started": False,
            "archive_member_paths_verified": False,
            "records_expected": len(rows),
            "records_in_checkpoint": total,
            "total_file_bytes_by_stat": total_file_bytes,
            "hashed_this_run": hashed_now,
            "reused_from_checkpoint": reused,
            "hash_failures": hash_failures,
            "missing_hashes": missing_hashes,
            "decode_failures": decode_failures,
            "groups": group_counts,
            "checkpoint_sqlite": str(db_path),
            "file_manifest_jsonl": str(manifest_path),
            "elapsed_seconds_this_run": round(time.time() - started, 2),
            "notes": [
                "SHA-256 covers the currently mounted file bytes only; it does not prove original archive or source provenance.",
                "Images are fully decoded one at a time after hashing; the input tree is never written.",
                "Checkpoint reuse assumes an immutable, identical mounted Dataset when path, size, and mtime match.",
            ],
        }
        temporary_summary = summary_path.with_suffix(summary_path.suffix + ".tmp")
        temporary_summary.write_text(json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
        os.replace(temporary_summary, summary_path)
        summary["summary_json"] = str(summary_path)
        return summary
    finally:
        db.close()


In [ ]:
# Stage 2: only after path-index cell succeeded; reuses `DIRE_ROOT` and `records`
integrity_result = verify_celebahq_records(
    root=DIRE_ROOT,
    records=records,
    workdir=Path('/kaggle/working'),
    checkpoint_every=250,
    progress_every=1000,
)
print('Integrity status:', integrity_result['status'])
print('Records:', integrity_result['records_in_checkpoint'], '/', integrity_result['records_expected'])
print('Hashed this run:', integrity_result['hashed_this_run'])
print('Reused checkpoint:', integrity_result['reused_from_checkpoint'])
print('SHA failures:', integrity_result['hash_failures'])
print('Decode failures:', integrity_result['decode_failures'])
print('SQLite checkpoint:', integrity_result['checkpoint_sqlite'])
print('JSONL manifest:', integrity_result['file_manifest_jsonl'])
print('Summary:', integrity_result['summary_json'])
